## Idea

This notebook is the first of the second training edition `#2_rolling_validation_12feat`. It prepares the dataset for a rolling validation strategy by making a 80/20 chronological split (v4).

<i> Note: Since it uses the same feature set as edition 1, notebooks that have to do with feature engineering or hyperparameter tuning will be limitted. </i>

---

### Imports

In [1]:
from pathlib import Path
import pandas as pd

### Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "data"

ALL_PROFILES_PATH = DATA_PATH / "profiles" / "all_profiles.parquet"

TRAIN_V4_PATH = DATA_PATH / "dvc" / "train_v4.parquet"
TEST_V4_PATH = DATA_PATH / "dvc" / "test_v4.parquet"

### Load profiles and make chronological train/test split

In [3]:
df = pd.read_parquet(ALL_PROFILES_PATH)

df["profile_date"] = pd.to_datetime(df["profile_date"])

print(df.shape)
print(df["profile_date"].min(), "→", df["profile_date"].max())

# Sort chronologically
df = df.sort_values("profile_date")

unique_days = sorted(df["profile_date"].unique())

# Reserve last 3 days for final testing
test_days = unique_days[-3:]
train_days = unique_days[:-3]

train_df = df[df["profile_date"].isin(train_days)].copy()
test_df = df[df["profile_date"].isin(test_days)].copy()

print("Train days:", train_days)
print("Test days:", test_days)

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label distribution:")
display(train_df["label"].value_counts(normalize=True))

print("\nTest label distribution:")
display(test_df["label"].value_counts(normalize=True))

(428800, 42)
2022-11-01 00:00:00 → 2022-11-15 00:00:00
Train days: [Timestamp('2022-11-01 00:00:00'), Timestamp('2022-11-02 00:00:00'), Timestamp('2022-11-03 00:00:00'), Timestamp('2022-11-04 00:00:00'), Timestamp('2022-11-05 00:00:00'), Timestamp('2022-11-06 00:00:00'), Timestamp('2022-11-07 00:00:00'), Timestamp('2022-11-08 00:00:00'), Timestamp('2022-11-09 00:00:00'), Timestamp('2022-11-10 00:00:00'), Timestamp('2022-11-11 00:00:00'), Timestamp('2022-11-12 00:00:00')]
Test days: [Timestamp('2022-11-13 00:00:00'), Timestamp('2022-11-14 00:00:00'), Timestamp('2022-11-15 00:00:00')]

Train shape: (343365, 42)
Test shape: (85435, 42)

Train label distribution:


label
0    0.942417
1    0.057583
Name: proportion, dtype: float64


Test label distribution:


label
0    0.942834
1    0.057166
Name: proportion, dtype: float64

### Save versioned datasets

In [4]:
train_df.to_parquet(TRAIN_V4_PATH, index=False)
test_df.to_parquet(TEST_V4_PATH, index=False)

print("Saved:")
print(TRAIN_V4_PATH)
print(TEST_V4_PATH)

Saved:
c:\Users\ali.farhat\Documents\simbox-project\data\dvc\train_v4.parquet
c:\Users\ali.farhat\Documents\simbox-project\data\dvc\test_v4.parquet


---
### Version Metadata

In [5]:
print("=" * 60)
print("Dataset Version: v4")
print("=" * 60)

print(f"Total profiles: {len(df):,}")
print(f"Train profiles: {len(train_df):,}")
print(f"Test profiles: {len(test_df):,}")

print("\nTrain period:")
print(train_df["profile_date"].min(), "→", train_df["profile_date"].max())

print("\nTest period:")
print(test_df["profile_date"].min(), "→", test_df["profile_date"].max())

print("\nTrain unique MSISDNs:", train_df["msisdn"].nunique())
print("Test unique MSISDNs:", test_df["msisdn"].nunique())

print("\nTrain fraud ratio:", round(train_df["label"].mean(), 4))
print("Test fraud ratio:", round(test_df["label"].mean(), 4))

print("\nFeatures:", len(train_df.columns))
print("\nSaved version:")
print("train_v4.parquet")
print("test_v4.parquet")

Dataset Version: v4
Total profiles: 428,800
Train profiles: 343,365
Test profiles: 85,435

Train period:
2022-11-01 00:00:00 → 2022-11-12 00:00:00

Test period:
2022-11-13 00:00:00 → 2022-11-15 00:00:00

Train unique MSISDNs: 31660
Test unique MSISDNs: 30619

Train fraud ratio: 0.0576
Test fraud ratio: 0.0572

Features: 42

Saved version:
train_v4.parquet
test_v4.parquet
